# Bài 10  · MLP và học biểu diễn
PGS.TS. Nguyễn Văn Hậu. Chạy lần lượt từ đầu. Các số của case dùng checkpoint đã chốt; ô cuối cho phép chủ động chạy lại toàn bộ. Dữ liệu test đã có lịch sử dùng trong .

In [ ]:
from pathlib import Path
import sys, json
R=Path.cwd()
if not (R/'source/mlp_lab.py').exists() and (R.parent/'source/mlp_lab.py').exists(): R=R.parent
assert (R/'source/mlp_lab.py').exists(), 'Open the notebook from the extracted lesson folder'
if (R/'.build/pydeps').exists(): sys.path.insert(0,str(R/'.build/pydeps'))
sys.path.insert(0,str(R/'source'))
import numpy as np
from mlp_lab import tiny_forward, finite, softmax, forward, bce
import sklearn
print('scikit-learn',sklearn.__version__)

## 1. Ảnh thật, ID và cách chia
ID nguồn và vai trò thí nghiệm khác nhau. Các giá trị pixel dùng uint8/255.

In [ ]:
D=np.load(R/'data/fashion_subset.npz'); F=np.load(R/'data/lab_arrays.npz')
A=json.loads((R/'results/case_audit.json').read_text(encoding='utf-8'))
assert not set(D['train_indices']) & set(D['validation_indices'])
for split in ['train','validation','test']: print(split,D[split+'_images'].shape)
i=int(F['display_positions'][0]); x=D['validation_images'][i].astype(float)/255
print('train:'+str(D['validation_indices'][i]),'validation',i,'pixel',D['validation_images'][i].reshape(28,28)[12,12])
assert D['validation_images'][i].reshape(28,28)[12,12]==203

## 2. XOR: khả năng biểu diễn với trọng số cho trước

In [ ]:
X=np.array([[0,0],[0,1],[1,0],[1,1]],float); yxor=np.array([0,1,1,0])
H=np.c_[np.maximum(X.sum(1),0),np.maximum(X.sum(1)-1,0)]
score=H@np.array([2.,-4.])-1
print('H=',H,'scores=',score,sep='\n')
assert np.array_equal((score>=0).astype(int),yxor)

## 3. Forward mạng nhỏ: cùng chín tham số

In [ ]:
theta=np.array([.5,-.4,.3,.8,.1,.1,.7,-.5,.2])
old=tiny_forward(theta)
for key in ['x','z','h','s','p','loss']: print(key,old[key])
assert np.isclose(old['s'],.204)

## 4. Gradient theo từng đường và kiểm độc lập bằng sai phân

In [ ]:
g=old['gradient']; gn=finite(lambda t:tiny_forward(t)['loss'],theta)
print('gradient',g,'numeric',gn,'max error',np.abs(g-gn).max(),sep='\n')
assert np.allclose(g,gn,atol=1e-9)
assert np.isclose(g[1],(old['p']-1)*theta[6]*1*.2)

## 5. Cập nhật đồng thời và forward lại
Các gradient cùng thuộc θ cũ. Loss giảm trong ví dụ này, nhãn vẫn là 1.

In [ ]:
eta=.1; new_theta=theta-eta*g; fresh=tiny_forward(new_theta)
print('old/new loss',old['loss'],fresh['loss'])
print('w12',theta[1],'->',new_theta[1])
assert fresh['loss']<old['loss']
assert (old['p']>=.5)==(fresh['p']>=.5)

## 6. Softmax, one-hot và gradient theo score
Ba score này là ví dụ toán, khác score của ảnh thật.

In [ ]:
scores=np.array([2.,1.,0.]); probs=softmax(scores); onehot=np.array([0,1,0])
loss=-np.log(probs[1]); delta=probs-onehot
print('p',probs,'CE',loss,'delta',delta)
assert np.isclose(probs.sum(),1) and np.isclose(delta.sum(),0)

## 7. Theo cùng ảnh qua checkpoint thật
Mảng batch dùng hàng là mẫu, coefs_ chuyển vị với công thức vector cột trên slide.

In [ ]:
xx=D['validation_images'][i:i+1].astype(float)/255
hh,ss,pp=forward(xx,F['W1'],F['b1'],F['W2'],F['b2'])
print('ID',D['validation_indices'][i],'hidden',hh.shape,'active',np.count_nonzero(hh))
print('scores',ss[0],'p',pp[0],sep='\n')
assert pp.argmax(1)[0]==F['validation_predictions'][i]
assert np.isclose(pp.sum(),1)

## 8. Confusion, checkpoint và hai chiều lỗi

In [ ]:
yv=D['validation_labels']; pred=F['validation_predictions']; base=F['validation_baseline_predictions']
cm=np.zeros((10,10),int); np.add.at(cm,(yv,pred),1)
print('correct',cm.trace(),'recall class6',cm[6,6]/cm[6].sum())
counts=[int(((base==yv)&(pred==yv)).sum()),int(((base!=yv)&(pred==yv)).sum()),int(((base==yv)&(pred!=yv)).sum()),int(((base!=yv)&(pred!=yv)).sum())]
print('both correct / corrected / new / both wrong',counts)
history=A['history'][0]['epochs']; selected=1+int(np.argmin([h['validation_ce'] for h in history]))
print('selected epoch',selected)
assert counts==[2433,165,67,335] and selected==34 and cm.trace()==2598

## 9. Đường học và tùy chọn chạy lại
CE dưới đây tính lại trên toàn tập cuối epoch, không gồm L2. Test dùng báo cáo sau quyết định và đã có lịch sử . Đổi RUN_FULL thành True chỉ khi muốn chạy lại cả ba seed và cập nhật kết quả cục bộ.

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(9,4))
ax.plot([h['epoch'] for h in history],[h['train_ce'] for h in history],label='train CE')
ax.plot([h['epoch'] for h in history],[h['validation_ce'] for h in history],label='validation CE')
ax.axvline(selected,color='gray',linestyle='--',label='selected checkpoint')
ax.set(xlabel='epoch',ylabel='mean cross-entropy'); ax.legend(); plt.show()
RUN_FULL=False
if RUN_FULL:
    import subprocess
    subprocess.run([sys.executable,str(R/'source/mlp_lab.py'),'--force'],check=True,cwd=R)
print('Test:',A['baseline_test']['correct'],A['mlp_test']['correct'],'/10000; benchmark history disclosed')